In [2]:
#Libraries
import os
import sys
import dask
import pandas as pd
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

In [3]:

#--------------------------------------------------------------
#Helper functions 
#--------------------------------------------------------------

def time_filter(data):
    winter_month = list([1,2,11,12])
    time_months = data['time'].dt.month
    time_mask = time_months.isin(winter_month)
    data = data.where(time_mask, drop=True)
    return data

def lat_lon_filter(data):
    lat_mask = ((data['lat'] >= 30) & (data['lat'] <= 80)) | ((data['lat'] >= -80) & (data['lat'] <= -30))
    lon_mask = (data['lon'] >= -80) | (data['lon'] <= 80)
    data = data.where(lat_mask & lon_mask, drop=True)
    return data

In [4]:
#--------------------------------------------------------------
#Input data 
#--------------------------------------------------------------
#Folder path
folder_path = "/wolke_scratch/dnikolo/ICON_Data/"
array_path = "/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_09_02/"
file = str(sys.argv[1])
file_path="/wolke_scratch/dnikolo/ICON_Data/atm_2d_remap_test.nc"
# file_path = folder_path + str(file)

In [5]:

from datetime import datetime, timedelta

def float_time_to_datetime(t):
    """
    Convert 'day as %Y%m%d.%f' float (e.g. 20080905.010417)
    to a Python datetime.
    """
    t = float(t)
    date_int = int(t)
    frac = t - date_int  # fraction of day

    year  = date_int // 10000
    month = (date_int % 10000) // 100
    day   = date_int % 100

    # fraction of day -> seconds (round to nearest second)
    seconds = int(round(frac * 24 * 3600))

    return datetime(year, month, day) + timedelta(seconds=seconds)

# Production

In [ ]:
#---------------------------------------------------------------
#Temperature range loop
#---------------------------------------------------------------
T_ranges = [(i,i+6) for i in range(273-6, 273-38, -6)]
folder_path="/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_03_03/remaped_ctp/"
base_dir = "/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_03_03/Filtered_Data/np"

for file in os.listdir(folder_path):
    # if file.startswith("test01_atm_3d"):

    # if file.startswith("test01_atm_3d_ml_20080901"):
    if file.startswith("remap_laf_whole_ryan_nudged_test"):
        file_path = folder_path + str(file)
    else:
        continue
    data = xr.open_dataset(
        file_path,
        chunks={'time': 1}
    )
    data = lat_lon_filter(data)
    #Load variables
    CPH = data['ctp']
    CTT = data['ctt']
    print(f"Processing file: {file}"
          f" with shape: {CPH.shape}")
    CPH_results = []
    CTT_results = []
    for T_range in T_ranges:
        #Temperature bounds
        T_min, T_max = T_range
        #Mask cloudy gridboxes within T range
        CTT_mask =  (CTT >= T_min) & (CTT < T_max)
    

        #Apply mask
        # CTT_results.append(CTT.where(CTT_mask,other=0))
        CPH_results.append(CPH.where(CTT_mask,other=0))
    with ProgressBar(dt=15.0):
        (
            # CTT_output,
            CPH_output

        ) = dask.compute(
            # CTT_results,
            CPH_results
            
        )
    for i, cph_temp_range in enumerate(CPH_output):
        # Assign attributes to DataArray otherwise it can not be converted to Dataset
        cph_temp_range = cph_temp_range.assign_attrs({"name": "cph","long_name":"Cloud Phase","units":"unitless"}).rename("cph_filtered")
        
        # convert times to datetimes
        # --- convert float time (YYYYMMDD + frac day) -> datetime64 ---
        t = cph_temp_range["time"].values.astype(float)
        ymd  = np.floor(t).astype(int)
        frac = t - ymd
        base = pd.to_datetime(ymd.astype(str), format="%Y%m%d")
        # Concvert to datetime64[ns] and add 30 seconds to avoid rounding issues when converting to minutes. Minute precision rounds down.
        dt = ((base + pd.to_timedelta(frac, unit="D")).to_numpy(dtype="datetime64[ns]") + np.timedelta64(30, 's')).astype('datetime64[m]')
        
        # Assign value for the time coordinate
        cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))

        # group by time: one dataset per time step
        times, dataset_list = zip(*cph_temp_range.to_dataset().groupby("time",squeeze=False))
        new_data_list = []
        for data,dt in zip(dataset_list,times):
            new_data_list.append(data.assign_coords(time=("time", [dt])))
            data = new_data_list[-1]
            data['time']= data['time'].assign_attrs({'standard_name': 'time',
                                                                        'axis': 'T'})
            data.time.encoding = {
                "units": "seconds since 2008-08-30 00:00:00",
                "calendar": "proleptic_gregorian",
            }
            data['lat']= data['lat'].assign_attrs({'_FillValue': np.nan ,
                                                                    'standard_name': 'latitude',
                                                                    'long_name': 'latitude',
                                                                    'units': 'degrees_north',
                                                                    'axis': 'Y'})
            data['lon']= data['lon'].assign_attrs({'_FillValue': np.nan ,
                                                                    'standard_name': 'longitude',
                                                                    'long_name': 'longitude',
                                                                    'units': 'degrees_east',
                                                                    'axis': 'X'})
        
        # ensure each dataset has `time` as a dimension (len=1), not a scalar coord
        dts = [t for t in times]

        # build one path per dataset in the desired format:
        # 2008/02/01/R_Agg_20_T_06_00_20080201000000.nc
        paths = []
        for dt in dts:
            ts = pd.Timestamp(dt)        # dt is numpy.datetime64
            subdir = ts.strftime("%Y/%m/%d")
            filename = f"R_Agg_30_T_{(273-T_ranges[i][0]):02}_{(273-T_ranges[i][1]):02}_{ts.strftime('%Y%m%d%H%M%S')}.nc"
            full_path = os.path.join(base_dir, subdir, filename)

            # make sure directory exists
            os.makedirs(os.path.dirname(full_path), exist_ok=True)
            paths.append(full_path)
        xr.save_mfdataset(new_data_list, paths)
    

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080830T000000Z.nc with shape: (96, 67, 240)
[########################################] | 100% Completed | 15.02 s


/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080831T000000Z.nc with shape: (96, 67, 240)
[########################################] | 100% Completed | 15.03 s


/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080901T000000Z.nc with shape: (96, 67, 240)
[########################################] | 100% Completed | 15.02 s


...

/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080907T000000Z.nc with shape: (96, 67, 240)
[########################################] | 100% Completed | 15.04 s


/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080908T000000Z.nc with shape: (1, 67, 240)
[########################################] | 100% Completed | 15.00 s


/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_24907/3952141717.py:75: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

In [20]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

# ---------------------------------------------------------------
# Temperature range loop
# ---------------------------------------------------------------
T_ranges = [(i, i + 6) for i in range(273 - 6, 273 - 38, -6)]


base_dir = "/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_03_03"

folder_path = os.path.join(base_dir, "remaped_ctp/")

filtered_base_dir = os.path.join(base_dir, "Filtered_Data","np")
resampled_base_dir = os.path.join(base_dir, "Resampled_Data","np")

# ---- variables you want in the mixed-range output ----
# Replace these with the actual variable names from your dataset
mixed_var_names = [
    "ctt"
]

# Mixed temperature bounds across all ranges
T_min_mixed = min(x for t in T_ranges for x in t)
T_max_mixed = max(x for t in T_ranges for x in t)


def convert_time_coord(da):
    """
    Convert float time coordinate (YYYYMMDD + fractional day)
    to datetime64[m], matching your existing logic.
    """
    t = da["time"].values.astype(float)
    ymd = np.floor(t).astype(int)
    frac = t - ymd
    base = pd.to_datetime(ymd.astype(str), format="%Y%m%d")
    dt = (
        (base + pd.to_timedelta(frac, unit="D")).to_numpy(dtype="datetime64[ns]")
        + np.timedelta64(30, "s")
    ).astype("datetime64[m]")
    return da.assign_coords(time=("time", dt))


def prepare_split_datasets(ds):
    """
    Split dataset into one dataset per timestep and assign coord attrs/encoding.
    """
    times, dataset_list = zip(*ds.groupby("time", squeeze=False))
    new_data_list = []

    for data, dt in zip(dataset_list, times):
        data = data.assign_coords(time=("time", [dt]))

        data["time"] = data["time"].assign_attrs({
            "standard_name": "time",
            "axis": "T"
        })
        data.time.encoding = {
            "units": "seconds since 2008-08-30 00:00:00",
            "calendar": "proleptic_gregorian",
        }

        data["lat"] = data["lat"].assign_attrs({
            "_FillValue": np.nan,
            "standard_name": "latitude",
            "long_name": "latitude",
            "units": "degrees_north",
            "axis": "Y"
        })

        data["lon"] = data["lon"].assign_attrs({
            "_FillValue": np.nan,
            "standard_name": "longitude",
            "long_name": "longitude",
            "units": "degrees_east",
            "axis": "X"
        })

        new_data_list.append(data)

    return list(times), new_data_list


for file in os.listdir(folder_path):
    if file.startswith("remap_laf_whole_ryan_nudged_test"):
        file_path = os.path.join(folder_path, file)
    else:
        continue

    data = xr.open_dataset(file_path, chunks={"time": 1})
    data = lat_lon_filter(data)

    # Load variables
    CPH = data["ctp"]
    CTT = data["ctt"]

    # Extra variables for the mixed-range files
    mixed_vars = [data[v] for v in mixed_var_names]

    print(f"Processing file: {file} with shape: {CPH.shape}")

    # ---------------------------------------------------------------
    # 1) Existing per-temperature-range output (unchanged behavior)
    # ---------------------------------------------------------------
    CPH_results = []
    CTT_results = []

    for T_range in T_ranges:
        T_min, T_max = T_range
        CTT_mask = (CTT >= T_min) & (CTT < T_max)

        CTT_results.append(CTT.where(CTT_mask, other=0))
        CPH_results.append(CPH.where(CTT_mask, other=0))

    # ---------------------------------------------------------------
    # 2) New mixed-range output (single mask, one file per time)
    # ---------------------------------------------------------------
    mixed_mask = (CTT >= T_min_mixed) & (CTT < T_max_mixed)
    mixed_results = [var.where(mixed_mask, other=0) for var in mixed_vars]

    with ProgressBar(dt=15.0):
        CTT_output, CPH_output, mixed_output = dask.compute(
            CTT_results,
            CPH_results,
            mixed_results
        )

    # ---------------------------------------------------------------
    # Write existing filtered CPH files (same as before)
    # ---------------------------------------------------------------
    for i, cph_temp_range in enumerate(CPH_output):
        cph_temp_range = cph_temp_range.assign_attrs({
            "name": "cph",
            "long_name": "Cloud Phase",
            "units": "unitless"
        }).rename("cph_filtered")

        cph_temp_range = convert_time_coord(cph_temp_range)

        cph_ds = cph_temp_range.to_dataset()
        times, new_data_list = prepare_split_datasets(cph_ds)

        paths = []
        for dt in times:
            ts = pd.Timestamp(dt)
            subdir = ts.strftime("%Y/%m/%d")
            filename = (
                f"R_Agg_30_T_{(273 - T_ranges[i][0]):02}_{(273 - T_ranges[i][1]):02}_"
                f"{ts.strftime('%Y%m%d%H%M%S')}.nc"
            )
            full_path = os.path.join(filtered_base_dir, subdir, filename)
            os.makedirs(os.path.dirname(full_path), exist_ok=True)
            paths.append(full_path)

        xr.save_mfdataset(new_data_list, paths)

    # ---------------------------------------------------------------
    # Write new mixed-range files (one file per timestep)
    # ---------------------------------------------------------------
    mixed_ds = xr.Dataset()

    for var_name, var_da in zip(mixed_var_names, mixed_output):
        # optional: rename to indicate filtering
        out_name = f"{var_name}_filtered"

        mixed_ds[out_name] = convert_time_coord(var_da).rename(out_name)

    times, new_data_list = prepare_split_datasets(mixed_ds)

    paths = []
    for dt in times:
        ts = pd.Timestamp(dt)
        subdir = ts.strftime("%Y/%m/%d")
        filename = f"R_Agg_30_{ts.strftime('%Y%m%d%H%M%S')}.nc"
        full_path = os.path.join(resampled_base_dir, subdir, filename)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        paths.append(full_path)

    xr.save_mfdataset(new_data_list, paths)

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080830T000000Z.nc with shape: (96, 67, 240)
[                                        ] | 0% Completed | 258.43 us

KeyboardInterrupt: 

In [ ]:
T_min_mixed = min(x for t in T_ranges for x in t)
T_max_mixed = max(x for t in T_ranges for x in t)
CTT_mask = (CTT >= T_min_mixed) & (CTT < T_max_mixed)
CTT_result = CTT.where(CTT_mask, other=0)
resample_properties = [CTT_result]
for i in range(len(resample_properties)):
    # Assign attributes to DataArray otherwise it can not be converted to Dataset
    resample_properties[i] = resample_properties[i].assign_attrs({"name": "ctt","long_name":"Cloud Top Temperature","units":"K"}).rename("ctt_filtered")

# convert times to datetimes
# --- convert float time (YYYYMMDD + frac day) -> datetime64 ---
t = resample_properties[0]["time"].values.astype(float)
ymd  = np.floor(t).astype(int)
frac = t - ymd
base = pd.to_datetime(ymd.astype(str), format="%Y%m%d")
# Concvert to datetime64[ns] and add 30 seconds to avoid rounding issues when converting to minutes. Minute precision rounds down.
dt = ((base + pd.to_timedelta(frac, unit="D")).to_numpy(dtype="datetime64[ns]") + np.timedelta64(30, 's')).astype('datetime64[m]')

for i in range(len(resample_properties)):
    # Assign value for the time coordinate
    property_resampled = property_resampled.assign_coords(time=("time", dt))



237

In [ ]:
# CTT_mask = (CTT >= T_min) & (CTT < T_max)
# for i, ctt_resampled

# Test

In [5]:
#---------------------------------------------------------------
#Temperature range loop
#---------------------------------------------------------------
T_ranges = [(0,400)]
folder_path="/wolke_scratch/dnikolo/ICON_Data/Nudged_run_cph_03_03/remaped_ctp/"
base_dir = "/wolke_scratch/dnikolo/ICON_Data/Testing/online_ct_data"
remap_method = "laf"
os.makedirs(base_dir, exist_ok=True)

for file in os.listdir(folder_path):
    # if file.startswith("test01_atm_3d"):

    # if file.startswith("test01_atm_3d_ml_20080901"):
    if file.startswith(f"remap_{remap_method}_whole_ryan_nudged_test"):
        file_path = folder_path + str(file)
    else:
        continue
    data = xr.open_dataset(
        file_path,
        chunks={'time': 1}
    )
    # data = lat_lon_filter(data)
    #Load variables
    CPH = data['ctp']
    CTT = data['ctt']
    print(f"Processing file: {file}"
          f" with shape: {CPH.shape}")
    CPH_results = []
    CTT_results = []
    for T_range in T_ranges:
        #Temperature bounds
        T_min, T_max = T_range
        #Mask cloudy gridboxes within T range
        CTT_mask =  (CTT >= T_min) & (CTT < T_max)
    

        #Apply mask
        CTT_results.append(CTT.where(CTT_mask,other=0))
        CPH_results.append(CPH.where(CTT_mask,other=0))
    with ProgressBar(dt=15.0):
        (
            CTT_output,
            CPH_output

        ) = dask.compute(
            CTT_results,
            CPH_results
            
        )
    # ---------------------------------------------------------------
    # Save one NetCDF per temperature range, containing BOTH ctt + cph
    # ---------------------------------------------------------------
    for i, (cph_temp_range, ctt_temp_range) in enumerate(zip(CPH_output, CTT_output)):
        # Name + attrs
        cph_temp_range = cph_temp_range.rename("cph_filtered").assign_attrs({
            "long_name": "Cloud Phase",
            "units": "unitless"
        })

        ctt_temp_range = ctt_temp_range.rename("ctt_filtered").assign_attrs({
            "long_name": "Cloud Top Temperature (filtered)",
            "units": "K"  # adjust if your temperature is not Kelvin
        })

        # --- convert float time (YYYYMMDD + fractional day) -> datetime64 ---
        # Assumes original time is encoded like 20080901.5 etc.
        t = cph_temp_range["time"].values.astype(float)
        ymd = np.floor(t).astype(int)
        frac = t - ymd
        base = pd.to_datetime(ymd.astype(str), format="%Y%m%d")

        # Convert to datetime64[ns], add 30 s to avoid minute-rounding issues, then round to minutes
        dt = (
            (base + pd.to_timedelta(frac, unit="D")).to_numpy(dtype="datetime64[ns]")
            + np.timedelta64(30, "s")
        ).astype("datetime64[m]")

        # Assign same time coordinate to both arrays
        cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
        ctt_temp_range = ctt_temp_range.assign_coords(time=("time", dt))

        # Build one dataset containing BOTH variables
        ds = xr.Dataset({
            "cph_filtered": cph_temp_range,
            "ctt_filtered": ctt_temp_range,
        })

        # ---- Coordinate attrs / encoding ----
        ds["time"] = ds["time"].assign_attrs({
            "standard_name": "time",
            "axis": "T",
        })
        ds["time"].encoding = {
            "units": "seconds since 2008-08-30 00:00:00",
            "calendar": "proleptic_gregorian",
        }

        ds["lat"] = ds["lat"].assign_attrs({
            "_FillValue": np.nan,
            "standard_name": "latitude",
            "long_name": "latitude",
            "units": "degrees_north",
            "axis": "Y",
        })

        ds["lon"] = ds["lon"].assign_attrs({
            "_FillValue": np.nan,
            "standard_name": "longitude",
            "long_name": "longitude",
            "units": "degrees_east",
            "axis": "X",
        })

        # Optional: variable encodings (compression)
        # ds["cph_filtered"].encoding = {"zlib": True, "complevel": 4, "_FillValue": np.nan}
        # ds["ctt_filtered"].encoding = {"zlib": True, "complevel": 4, "_FillValue": np.nan}

        # ---- Output file ----
        start_ts = pd.Timestamp(ds.time.values[0])
        end_ts = pd.Timestamp(ds.time.values[-1])

        filename = (
            f"Online_ct_{remap_method}_all_"
            f"{start_ts.strftime('%Y%m%d%H%M%S')}.nc"
        )
        full_path = os.path.join(base_dir, filename)

        ds.to_netcdf(full_path)

Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080830T000000Z.nc with shape: (96, 120, 240)
[########################################] | 100% Completed | 15.01 s
Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080831T000000Z.nc with shape: (96, 120, 240)
[                                        ] | 0% Completed | 207.43 us

/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting

[########################################] | 100% Completed | 15.01 s
Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080901T000000Z.nc with shape: (96, 120, 240)
[                                        ] | 0% Completed | 372.61 us

/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting

[########################################] | 100% Completed | 15.01 s
Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080902T000000Z.nc with shape: (96, 120, 240)
[                                        ] | 0% Completed | 248.42 us

...

/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting

[########################################] | 100% Completed | 15.01 s
Processing file: remap_laf_whole_ryan_nudged_test_v0203_ctp_2d_ml_20080908T000000Z.nc with shape: (1, 120, 240)
[                                        ] | 0% Completed | 203.97 us

/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting

[########################################] | 100% Completed | 15.00 s


/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_9018/2556096193.py:79: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting